# 方向二：正式版 DEA–KMeans 省级医疗资源配置效能评估

**角色定位：成员6（算法架构师）**

本 Notebook 使用成员4最新交接的 `Final_Model_Dataset_Strict.csv`，对中国 31 个省份在 **2020、2021、2022** 三个年份的医疗卫生资源配置效率进行正式建模。

本版与之前的 Demo 版相比，核心变化有三点：

1. **删除伪产出**，改为使用真实投入与真实健康产出；
2. **正式运行 DEA**，主模型采用 **BCC 输入导向**，辅助报告 CCR 与规模效率；
3. **对 2022 年结果做 4 类 K-Means 聚类**，输出正式结果表，供成员5制图和文案组写作使用。

> 说明：根据交接要求，非期望产出（孕产妇死亡率、婴儿死亡率）需要先做正向化处理，再进入传统 DEA。

## 1. 导入所需库并配置环境

本部分导入数据处理、DEA 求解与聚类分析所需的核心库，并统一设置中文绘图环境。

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.optimize import linprog
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "SimSun"]
mpl.rcParams["axes.unicode_minus"] = False

## 2. 设置路径与核心参数

这里统一设置数据路径、输出路径、分析年份和聚类参数。

In [ ]:
BASE_DIR = Path(r"E:\2026_BigData_Project")
DATA_PATH = BASE_DIR / "data" / "processed" / "Final_Model_Dataset_Strict.csv"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS_TO_RUN = [2020, 2021, 2022]
LATEST_YEAR = 2022
RANDOM_SEED = 42
N_CLUSTERS = 4

INPUT_COLS = [
    "每千人口床位数(张/千人)",
    "卫生技术人员总数(人)",
    "人均医疗卫生财政支出(元)",
]

DESIRABLE_OUTPUT_COLS = [
    "预期寿命 (岁)",
]

UNDESIRABLE_OUTPUT_COLS = [
    "孕产妇死亡率 (1/10 万)",
    "婴儿死亡率 (‰)",
]

## 3. 读取数据并检查关键字段

交接单说明，严格建模表是 **31省 × 3年 = 93行** 的纯净建模面板。  
为了防止后续 DEA 报错，先检查关键字段是否齐全且不存在缺失值。

In [ ]:
def load_data(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"找不到数据文件：{csv_path}")

    df = pd.read_csv(csv_path)

    required_cols = ["地区", "年份"] + INPUT_COLS + DESIRABLE_OUTPUT_COLS + UNDESIRABLE_OUTPUT_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"缺少必要字段：{missing}")

    if df[required_cols].isna().any().any():
        missing_counts = df[required_cols].isna().sum()
        raise ValueError(f"关键字段存在缺失值：\n{missing_counts[missing_counts > 0]}")

    return df

df = load_data(DATA_PATH)
print(df.shape)
df.head()

## 4. 对非期望产出做正向化处理

传统 DEA 更适合处理“越大越好”的产出变量。  
因此，对以下非期望产出先做倒数变换：

- 孕产妇死亡率
- 婴儿死亡率

处理公式为：

\[
y^* = \frac{1}{y}
\]

这样变换后，原本“越低越好”的死亡率就会转成“越高越好”的正向指标。

In [ ]:
def prepare_outputs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-9
    out["inv_孕产妇死亡率"] = 1.0 / (out["孕产妇死亡率 (1/10 万)"].astype(float) + eps)
    out["inv_婴儿死亡率"] = 1.0 / (out["婴儿死亡率 (‰)"].astype(float) + eps)
    return out

df = prepare_outputs(df)
df[["地区", "年份", "预期寿命 (岁)", "孕产妇死亡率 (1/10 万)", "婴儿死亡率 (‰)", "inv_孕产妇死亡率", "inv_婴儿死亡率"]].head()

## 5. DEA 求解器：输入导向包络模型

本研究主模型采用 **BCC 输入导向 DEA**。其含义是：

> 在保持健康产出不低于当前水平的前提下，考察各省投入最多可以压缩到原来的多少比例。

同时，为了拆分规模因素，辅助报告 **CCR 效率** 和 **规模效率**。

In [ ]:
def solve_dea_input_oriented(X: np.ndarray, Y: np.ndarray, rts: str = "bcc") -> np.ndarray:
    n, m = X.shape
    _, s = Y.shape

    eff_scores = np.full(n, np.nan)

    for o in range(n):
        c = np.zeros(n + 1)
        c[-1] = 1.0

        A_ub = []
        b_ub = []

        for i in range(m):
            row = np.zeros(n + 1)
            row[:n] = X[:, i]
            row[-1] = -X[o, i]
            A_ub.append(row)
            b_ub.append(0.0)

        for r in range(s):
            row = np.zeros(n + 1)
            row[:n] = -Y[:, r]
            A_ub.append(row)
            b_ub.append(-Y[o, r])

        A_eq = None
        b_eq = None

        if rts.lower() == "bcc":
            A_eq = [np.r_[np.ones(n), 0.0]]
            b_eq = [1.0]

        bounds = [(0, None)] * n + [(0, 1)]

        res = linprog(
            c=c,
            A_ub=np.array(A_ub, dtype=float),
            b_ub=np.array(b_ub, dtype=float),
            A_eq=np.array(A_eq, dtype=float) if A_eq is not None else None,
            b_eq=np.array(b_eq, dtype=float) if b_eq is not None else None,
            bounds=bounds,
            method="highs"
        )

        if not res.success:
            raise RuntimeError(f"DEA 求解失败（DMU={o}, rts={rts}）：{res.message}")

        eff_scores[o] = float(res.x[-1])

    return eff_scores

## 6. 对单一年份运行正式 DEA

投入项选择：

- 每千人口床位数
- 卫生技术人员总数
- 人均医疗卫生财政支出

产出项选择：

- 预期寿命
- 正向化后的孕产妇死亡率
- 正向化后的婴儿死亡率

In [ ]:
def run_dea_for_one_year(df_year: pd.DataFrame) -> pd.DataFrame:
    result = df_year.copy().reset_index(drop=True)

    X = result[INPUT_COLS].astype(float).to_numpy()
    Y = result[
        DESIRABLE_OUTPUT_COLS + ["inv_孕产妇死亡率", "inv_婴儿死亡率"]
    ].astype(float).to_numpy()

    X_scaled = X.copy()
    Y_scaled = Y.copy()

    for j in range(X_scaled.shape[1]):
        scale = np.nanmedian(np.abs(X_scaled[:, j]))
        if scale and scale > 0:
            X_scaled[:, j] = X_scaled[:, j] / scale

    for j in range(Y_scaled.shape[1]):
        scale = np.nanmedian(np.abs(Y_scaled[:, j]))
        if scale and scale > 0:
            Y_scaled[:, j] = Y_scaled[:, j] / scale

    result["DEA_CCR"] = solve_dea_input_oriented(X_scaled, Y_scaled, rts="ccr")
    result["DEA_BCC"] = solve_dea_input_oriented(X_scaled, Y_scaled, rts="bcc")
    result["规模效率"] = (result["DEA_CCR"] / result["DEA_BCC"]).clip(upper=1.0)

    return result

## 7. 循环运行 2020–2022 三年 DEA

这一步将三年的 DEA 结果拼接起来，便于后续进行：

- 年度比较
- 三年均值稳健性分析
- 2022 年聚类与最终可视化

In [ ]:
def run_dea_all_years(df: pd.DataFrame, years: list[int]) -> pd.DataFrame:
    frames = []
    for year in years:
        sub = df[df["年份"] == year].copy()
        one_year_result = run_dea_for_one_year(sub)
        frames.append(one_year_result)
    return pd.concat(frames, ignore_index=True)

dea_all = run_dea_all_years(df, YEARS_TO_RUN)
dea_all[["地区", "年份", "DEA_CCR", "DEA_BCC", "规模效率"]].head(10)

## 8. 查看三年 DEA 结果概况

先从整体上查看三年 DEA 效率的均值、最小值和最大值，判断模型是否已经正常跑通。

In [ ]:
dea_all.groupby("年份")[["DEA_CCR", "DEA_BCC", "规模效率"]].agg(["mean", "min", "max"])

## 9. 对 2022 年结果进行 K-Means 聚类

根据交接要求，最终需要输出 31 省的 **最新真实 DEA 效率得分和 4 类聚类标签**，供成员5画地图和雷达图。

这里选取 2022 年进行正式聚类，使用以下特征：

- DEA_BCC
- 规模效率
- 3个投入指标
- 预期寿命
- 正向化后的两类死亡率

聚类前统一做标准化处理。

In [ ]:
def run_kmeans_for_latest_year(df_latest: pd.DataFrame):
    result = df_latest.copy().reset_index(drop=True)

    feature_cols = [
        "DEA_BCC",
        "规模效率",
        "每千人口床位数(张/千人)",
        "卫生技术人员总数(人)",
        "人均医疗卫生财政支出(元)",
        "预期寿命 (岁)",
        "inv_孕产妇死亡率",
        "inv_婴儿死亡率",
    ]

    scaler = StandardScaler()
    X_std = scaler.fit_transform(result[feature_cols].astype(float))

    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=30)
    result["Cluster"] = kmeans.fit_predict(X_std)

    profile = result.groupby("Cluster")[feature_cols].mean().reset_index()

    profile["投入综合均值_z"] = StandardScaler().fit_transform(
        profile[[
            "每千人口床位数(张/千人)",
            "卫生技术人员总数(人)",
            "人均医疗卫生财政支出(元)"
        ]].mean(axis=1).to_frame()
    ).ravel()

    profile["效率综合均值_z"] = StandardScaler().fit_transform(
        profile[[
            "DEA_BCC",
            "规模效率",
            "预期寿命 (岁)",
            "inv_孕产妇死亡率",
            "inv_婴儿死亡率"
        ]].mean(axis=1).to_frame()
    ).ravel()

    name_map = {}
    for _, row in profile.iterrows():
        c = int(row["Cluster"])
        input_level = "高投入" if row["投入综合均值_z"] >= 0 else "低投入"
        eff_level = "高效率" if row["效率综合均值_z"] >= 0 else "低效率"

        if input_level == "高投入" and eff_level == "高效率":
            name_map[c] = "高投入高效率优势区"
        elif input_level == "高投入" and eff_level == "低效率":
            name_map[c] = "高投入低效率洼地区"
        elif input_level == "低投入" and eff_level == "高效率":
            name_map[c] = "低投入高效率紧凑区"
        else:
            name_map[c] = "低投入低效率短板区"

    result["Cluster_Name"] = result["Cluster"].map(name_map)
    profile["Cluster_Name"] = profile["Cluster"].map(name_map)

    return result, profile

latest_2022 = dea_all[dea_all["年份"] == LATEST_YEAR].copy()
clustered_2022, cluster_profile = run_kmeans_for_latest_year(latest_2022)

clustered_2022[["地区", "DEA_BCC", "DEA_CCR", "规模效率", "Cluster", "Cluster_Name"]].sort_values("DEA_BCC", ascending=False).head(15)

## 10. 查看 2022 年聚类画像

下面查看四类省份在核心指标上的均值，用于后续命名与结果解读。

In [ ]:
cluster_profile

## 11. 计算三年平均结果表

为了增强结论稳健性，再额外生成一张 **2020–2022 三年平均效率表**，可在论文附录中使用。

In [ ]:
def build_three_year_mean_table(df_all_years: pd.DataFrame) -> pd.DataFrame:
    agg_cols = [
        "DEA_CCR", "DEA_BCC", "规模效率",
        "每千人口床位数(张/千人)", "卫生技术人员总数(人)",
        "人均医疗卫生财政支出(元)", "预期寿命 (岁)",
        "孕产妇死亡率 (1/10 万)", "婴儿死亡率 (‰)",
        "inv_孕产妇死亡率", "inv_婴儿死亡率",
    ]

    return (
        df_all_years.groupby("地区", as_index=False)[agg_cols]
        .mean()
        .sort_values("DEA_BCC", ascending=False)
        .reset_index(drop=True)
    )

mean_3y = build_three_year_mean_table(dea_all)
mean_3y.head(15)

## 12. 导出正式结果文件

根据交接要求，至少需要导出：

- `DEA_Scores_By_Year.csv`
- `DEA_Scores_Clusters_Final.csv`
- `DEA_Scores_Clusters_3YearMean.csv`
- `Cluster_Profile_2022.csv`

In [ ]:
out1 = OUTPUT_DIR / "DEA_Scores_By_Year.csv"
out2 = OUTPUT_DIR / "DEA_Scores_Clusters_Final.csv"
out3 = OUTPUT_DIR / "DEA_Scores_Clusters_3YearMean.csv"
out4 = OUTPUT_DIR / "Cluster_Profile_2022.csv"

dea_all.sort_values(["年份", "地区"]).to_csv(out1, index=False, encoding="utf-8-sig")
clustered_2022.sort_values("DEA_BCC", ascending=False).to_csv(out2, index=False, encoding="utf-8-sig")
mean_3y.to_csv(out3, index=False, encoding="utf-8-sig")
cluster_profile.to_csv(out4, index=False, encoding="utf-8-sig")

print(out1)
print(out2)
print(out3)
print(out4)

## 13. 阶段性结论与下一步任务

截至目前，本正式版 Notebook 已完成以下任务：

1. 使用真实投入与真实健康结果指标替代 Demo 阶段的伪产出；
2. 对 2020–2022 三年分别运行 DEA，得到 CCR、BCC 和规模效率；
3. 对 2022 年做 4 类 K-Means 聚类，并形成正式结果表；
4. 导出可供成员5和文案组直接使用的结果文件。

下一步你还需要完成两件事：

- 写一份 `成员6_算法模型与结果解读.docx`
- 根据聚类画像，对 4 个 Cluster 做“人话命名”和特征解读，供论文正文直接使用